# Systematic Ablation Studies

Every mechanistic interpretability investigation starts with **ablation**:
remove components and measure the impact. This notebook demonstrates
a structured framework for running, aggregating, and interpreting
ablation experiments.

This notebook covers the `irtk.ablation_study` module.

## Why This Matters

Systematic ablation studies remove components one at a time (or in combination) to measure their importance. Layer-by-layer ablation, head importance matrices, and double-ablation interaction tests reveal which components are necessary and how they depend on each other.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import ablation_study

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Layer-by-Layer Ablation

Which layers are most important for the prediction?

In [ ]:
prompts = [
    "The Eiffel Tower is located in",
    "The capital of France is",
    "London is the capital of",
]
token_seqs = [model.to_tokens(p) for p in prompts]
paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

# Attention ablation
attn_result = ablation_study.layer_by_layer_ablation(model, token_seqs, metric, component="attn")
mlp_result = ablation_study.layer_by_layer_ablation(model, token_seqs, metric, component="mlp")

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(model.cfg.n_layers)
ax.bar(x - 0.2, attn_result['effects'], 0.4, yerr=attn_result['std'],
       label='Attention', color='steelblue', capsize=3)
ax.bar(x + 0.2, mlp_result['effects'], 0.4, yerr=mlp_result['std'],
       label='MLP', color='coral', capsize=3)
ax.set_xlabel('Layer')
ax.set_ylabel('Metric change when ablated')
ax.set_title('Layer Importance: Attention vs MLP')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Head Importance Matrix

Which specific attention heads matter for the prediction?

In [ ]:
result = ablation_study.head_importance_matrix(model, token_seqs, metric)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(result['matrix'], cmap='RdBu_r', aspect='auto')
ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_title('Per-Head Ablation Effect')
plt.colorbar(im, ax=ax, label='Metric change')
plt.tight_layout()
plt.show()

# Top heads
summary = ablation_study.ablation_summary(
    result['matrix'],
    labels=[f'L{l}H{h}' for l in range(model.cfg.n_layers) for h in range(model.cfg.n_heads)],
    top_k=10
)
print(f"Top 10 most important heads:")
for name, effect in summary['top_components']:
    print(f"  {name}: {effect:+.4f}")
print(f"\nGini coefficient: {summary['gini_coefficient']:.3f} (higher = more concentrated)")

## 3. Position Sensitivity

Which input positions carry the most information for the prediction?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
token_strs = [tokenizer.decode([int(t)]) for t in tokens]

result = ablation_study.position_sensitivity(model, tokens, metric)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['red' if i == result['most_important_pos'] else 'steelblue'
          for i in range(len(tokens))]
ax.bar(range(len(tokens)), np.abs(result['effects']), color=colors)
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(token_strs, rotation=45, ha='right')
ax.set_ylabel('|Effect| when position ablated')
ax.set_title('Position Sensitivity')
plt.tight_layout()
plt.show()

print(f"Most important position: {result['most_important_pos']} "
      f"('{token_strs[result['most_important_pos']]}')")
print(f"Clean metric: {result['clean_metric']:.4f}")

## 4. Double Ablation Interaction

Do components work together? Joint ablation reveals
synergy (jointly necessary) and redundancy (individually sufficient).

In [ ]:
attn_hooks = [f"blocks.{i}.hook_attn_out" for i in range(model.cfg.n_layers)]
mlp_hooks = [f"blocks.{i}.hook_mlp_out" for i in range(model.cfg.n_layers)]

result = ablation_study.double_ablation_interaction(
    model, tokens, metric, attn_hooks, mlp_hooks
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(result['joint_effects'], cmap='RdBu_r', aspect='auto')
axes[0].set_xlabel('MLP layer')
axes[0].set_ylabel('Attention layer')
axes[0].set_title('Joint Ablation Effects')
plt.colorbar(im, ax=axes[0])

im = axes[1].imshow(result['interaction_matrix'], cmap='RdBu_r', aspect='auto')
axes[1].set_xlabel('MLP layer')
axes[1].set_ylabel('Attention layer')
axes[1].set_title('Interaction Effects (+ = synergistic)')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 5. Ablation Summary

Aggregate raw results into actionable findings.

In [ ]:
# Summarize the head importance matrix
head_labels = [f'L{l}H{h}' for l in range(model.cfg.n_layers)
               for h in range(model.cfg.n_heads)]
head_result = ablation_study.head_importance_matrix(model, token_seqs, metric)
summary = ablation_study.ablation_summary(head_result['matrix'], labels=head_labels)

print(f"Ablation Summary:")
print(f"  Mean |effect|: {summary['mean_effect']:.4f}")
print(f"  Max |effect|:  {summary['max_effect']:.4f}")
print(f"  Gini coeff:    {summary['gini_coefficient']:.4f}")
print(f"  N important:   {summary['n_important']} / {len(head_labels)}")
print(f"\nTop 5 components:")
for name, effect in summary['top_components'][:5]:
    print(f"  {name}: {effect:+.4f}")

## Summary

| Function | Purpose |
|----------|--------|
| `layer_by_layer_ablation()` | Ablate each layer in turn, aggregate across prompts |
| `head_importance_matrix()` | Per-head ablation effect matrix |
| `position_sensitivity()` | Per-position importance via embedding ablation |
| `double_ablation_interaction()` | Joint ablation to detect synergy/redundancy |
| `ablation_summary()` | Aggregate effects into top-k, Gini, summary stats |